# B3 local-refinement training: multi-seed uncertainty

The already-trained `b3_training_local_refine/checkpoint_50000.pt` is a single training run -- its weight initialization used PyTorch's default, unseeded global RNG (confirmed directly: `train_B3.py` has no `--seed` argument and never calls `torch.manual_seed`; the training *data* sampling is already deterministic, tied to the iteration number, so only weight init varies run to run).

This notebook trains **two more** local-refinement runs, identical in every other setting (same architecture, same 50,000 iterations, same `local_refine_n_sub=10`, same reused `input_norm.json`), with explicit `torch.manual_seed(2)` and `torch.manual_seed(3)`. Combined with the existing run (implicitly "seed 1"), this gives a genuine 3-point sample for mean ± std across training seeds, on: regional Cauchy-stress error (converged, n_sub=10), combined displacement error, energy error, reaction-moment error.

**Cost**: each run is the same configuration as the existing one, which took 7,627.5s (~2h7m) on an A100. Two runs ≈ 4h14m of GPU time total (run the two training cells back to back in this one notebook, or in two separate Colab sessions in parallel if you want wall-clock under ~2h15m).

In [ ]:
import os, subprocess, sys, time

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'timm', 'h5py', 'jax', 'tqdm'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)
for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

R = '/content/drive/MyDrive/pfem_run'
RUN4_DIR = f'{R}/b3_training_normalized'
print('Ready.')

In [ ]:
def train_and_evaluate_seed(seed):
    """Trains one local-refinement run with an explicit torch.manual_seed,
    then runs both the regional-stress convergence sweep and the global
    QoI/Jacobian check -- identical settings/code paths to the original
    b3_training_local_refine run, only the seed and output dir differ."""
    import shutil
    from omar_pfem.train_B3 import get_args, train, DEFAULT_RESOLUTION

    run_dir = f'{R}/b3_training_local_refine_seed{seed}'
    os.makedirs(run_dir, exist_ok=True)

    norm_src = f'{RUN4_DIR}/input_norm.json'
    norm_dst = f'{run_dir}/input_norm.json'
    assert os.path.exists(norm_src), f'baseline input_norm.json not found: {norm_src}'
    shutil.copy(norm_src, norm_dst)

    args = get_args([
        '--n_iters', '50000',
        '--batch_size', '8',
        '--log_every', '500',
        '--ckpt_every', '50000',
        '--output_dir', run_dir,
        '--normalize_inputs', '1',
        '--local_refine_region', '1',
        '--local_refine_n_sub', '10',
    ])

    print(f'\n{"="*90}\nSEED {seed}: training (50,000 iters, local_refine_n_sub=10, same config '
          f'as the original run, only the weight-init seed differs)\n{"="*90}')
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    model = train(args, device)
    elapsed = time.time() - t0
    print(f'SEED {seed}: training finished in {elapsed:.1f}s ({elapsed/args.n_iters:.4f}s/iter).')

    checkpoint = f'{run_dir}/checkpoint_{args.n_iters}.pt'
    dataset = f'{R}/b3_dataset_clean_holdout/dataset.h5'

    print(f'\nSEED {seed}: regional-stress convergence sweep...')
    sys.argv = [
        'evaluate_B3_region_local_refine.py',
        '--checkpoint', checkpoint, '--dataset', dataset,
        '--out_json', f'{run_dir}/region_local_refine_convergence.json',
        '--n_sub_sweep', '2', '4', '6', '8', '10', '12',
    ]
    from omar_pfem.evaluate_B3_region_local_refine import main as eval_region_main
    eval_region_main()

    print(f'\nSEED {seed}: global QoIs + Jacobian check...')
    sys.argv = [
        'evaluate_B3_global_qois_and_jacobian.py',
        '--checkpoint', checkpoint, '--dataset', dataset,
        '--out_json', f'{run_dir}/global_qois_and_jacobian.json',
    ]
    from omar_pfem.evaluate_B3_global_qois_and_jacobian import main as eval_global_main
    eval_global_main()

    return run_dir, elapsed

## Seed 2 (~2h7m)

In [ ]:
run_dir_2, elapsed_2 = train_and_evaluate_seed(2)

## Seed 3 (~2h7m)

In [ ]:
run_dir_3, elapsed_3 = train_and_evaluate_seed(3)

## Aggregate: mean ± std across all three seeds (copy this cell's output back)

Seed 1 is the already-existing `b3_training_local_refine` run (its region-stress sweep and global-QoI JSON already exist on Drive from earlier).

In [ ]:
import json
import numpy as np

run_dirs = {
    'seed1 (original)': f'{R}/b3_training_local_refine',
    'seed2': f'{R}/b3_training_local_refine_seed2',
    'seed3': f'{R}/b3_training_local_refine_seed3',
}

stress_converged, disp_combined, energy_err, reaction_err = [], [], [], []
per_seed = {}
for name, d in run_dirs.items():
    with open(f'{d}/region_local_refine_convergence.json') as f:
        region = json.load(f)
    with open(f'{d}/global_qois_and_jacobian.json') as f:
        glob = json.load(f)
    n_sub10 = next(r for r in region['sweep'] if r['n_sub'] == 10)
    stress = n_sub10['pooled_frobenius_rel_error']
    disp = glob['displacement']['mean_rel_L2_combined']
    energy = glob['energy_pooled_rms_rel']
    reaction = glob['reaction_moment_y_pooled_rms_rel']
    per_seed[name] = {'stress': stress, 'displacement': disp, 'energy': energy, 'reaction': reaction}
    stress_converged.append(stress)
    disp_combined.append(disp)
    energy_err.append(energy)
    reaction_err.append(reaction)

print('Per-seed results:')
print(json.dumps(per_seed, indent=2))

print('\n=== MEAN +/- STD ACROSS 3 SEEDS ===')
for name, vals in [('Regional Cauchy-stress (n_sub=10)', stress_converged),
                    ('Displacement (combined)', disp_combined),
                    ('Energy (pooled RMS)', energy_err),
                    ('Reaction moment (pooled RMS)', reaction_err)]:
    arr = np.array(vals)
    print(f'{name}: {arr.mean()*100:.2f}% +/- {arr.std()*100:.2f}%  (values: {[f"{v*100:.2f}%" for v in vals]})')